# AKT — kiểm CUDA tổng hợp trên Colab trước Modal

Ưu tiên Colab free cho các kiểm CUDA dùng chung, giữ credits Modal cho pilot/training và kiểm môi trường riêng khi cần. Chọn Runtime → Change runtime type → GPU khi sẵn sàng chạy gói kiểm nhỏ, upload source bundle theo `docs/akt-colab-synthetic-runbook.md`. Không dùng SSH/tunnel hoặc lách quota.

Dữ liệu tổng hợp400targets, một optimizer step. Bộ CPU portability phải đạt trước cell GPU; tải ZIP kết quả ngay sau lượt chạy. Đây là kiểm tương thích thiết bị và artifact lifecycle, chưa là throughput full-fold, mid-epoch CUDA resume hoặc Gate4. Kiểm riêng runtime/Volume Modal vẫn cần sau khi Colab đạt. Không có runtime Colab nào đã được kết nối chỉ bằng việc chuẩn bị notebook này.


In [ ]:
CPU_PORTABILITY_OK = False
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile, io, uuid, os, sys, subprocess, importlib.util
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Chỉ upload một ZIP source bundle')
name, data = next(iter(uploaded.items()))
if len(data) > 20 * 1024 * 1024:
    raise ValueError('Bundle vượt20MiB')
ROOT = Path('/content') / ('kltn-synthetic-' + uuid.uuid4().hex)
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)) or sum(x.file_size for x in archive.infolist()) > 50 * 1024 * 1024:
        raise ValueError('Archive duplicate hoặc vượt50MiB expanded')
    manifest = json.loads(archive.read('BUNDLE_MANIFEST.json'))
    if (manifest.get('schema_version'), manifest.get('scope')) not in {(1,'source_and_synthetic_tests_only'),(2,'source_and_bounded_test_fixtures')}:
        raise ValueError('Sai bundle schema/scope')
    if set(names) != set(manifest['files']) | {'BUNDLE_MANIFEST.json'}:
        raise ValueError('Bundle thiếu/thừa file')
    for path, expected in manifest['files'].items():
        relative = PurePosixPath(path)
        if relative.is_absolute() or '..' in relative.parts or '\\' in path or ':' in path:
            raise ValueError('Unsafe archive path')
        content = archive.read(path)
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Hash mismatch ' + path)
        destination = ROOT.joinpath(*relative.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
(ROOT / 'BUNDLE_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
(ROOT / 'BUNDLE_SHA256.txt').write_text(hashlib.sha256(data).hexdigest() + '\n')
print('Source snapshot', ROOT, 'SHA256', hashlib.sha256(data).hexdigest())


In [ ]:
# Venv riêng dưới project, kế thừa PyTorch runtime; chỉ cài thiếu vào venv.
subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(ROOT / '.venv')],check=True,timeout=120)
AKT_PY = str(ROOT / '.venv/bin/python')
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Runtime chưa có PyTorch, cần chuẩn bị image trước')
packages = {'numpy':'numpy', 'pandas':'pandas', 'duckdb':'duckdb',
            'sklearn':'scikit-learn', 'psutil':'psutil', 'pytest':'pytest', 'networkx':'networkx'}
probe = 'import importlib.util,json; print(json.dumps([p for m,p in ' + repr(packages) + '.items() if importlib.util.find_spec(m) is None]))'
missing = json.loads(subprocess.check_output([AKT_PY,'-c',probe],text=True))
if missing:
    subprocess.run([AKT_PY, '-m', 'pip', 'install', *missing], check=True, timeout=300)
upstream = ROOT / 'artifacts/akt/dependencies/upstream'
upstream.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(['git','clone','--no-checkout','https://github.com/arghosh/AKT.git',str(upstream)],check=True,timeout=180)
subprocess.run(['git','-C',str(upstream),'checkout','--detach','01b7e8d7d7c862cabc9467108e3e7f2555653a88'],check=True,timeout=60)
ENV = dict(os.environ, PYTHONPATH=str(ROOT / 'src'), PYTHONDONTWRITEBYTECODE='1')
OUTPUT = ROOT / 'colab-evidence'
OUTPUT.mkdir()
(OUTPUT / 'BUNDLE_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
(OUTPUT / 'BUNDLE_SHA256.txt').write_text(hashlib.sha256(data).hexdigest() + '\n')


In [ ]:
# CPU portability suite, timeout10phút; không cấp quyền chạy GPU từ exit0 đơn thuần.
CPU_PORTABILITY_OK = False
with (OUTPUT / 'cpu-tests.log').open('w') as log:
    result = subprocess.run([AKT_PY,'-B','-m','pytest','tests/akt','tests/shared_kt','tests/gam_kt',
                             '-q','-p','no:cacheprovider','-k','not test_native_akt_import_cpu and not test_actual_frozen_assignments_are_unchanged_and_student_disjoint',
                             '--basetemp',str(ROOT / 'pytest-temp'),'--junitxml',str(OUTPUT / 'cpu-tests.xml')],
                            cwd=ROOT,env=dict(ENV,CUDA_VISIBLE_DEVICES=''),stdout=log,stderr=subprocess.STDOUT,timeout=600)
print((OUTPUT / 'cpu-tests.log').read_text()[-6000:])
result.check_returncode()
import xml.etree.ElementTree as ET
suites = list(ET.parse(OUTPUT / 'cpu-tests.xml').getroot().iter('testsuite'))
counts = {key:sum(int(s.attrib.get(key,'0')) for s in suites) for key in ('tests','failures','errors','skipped')}
if counts['tests'] <= 0 or any(counts[key] for key in ('failures','errors','skipped')):
    raise RuntimeError('CPU portability yêu cầu suite không rỗng, không fail/error/skip; không chạy GPU')
(OUTPUT / 'cpu-acceptance.json').write_text(json.dumps({'status':'PASS','counts':counts,
    'timeout_seconds':600,'explicit_deselections':['test_native_akt_import_cpu','test_actual_frozen_assignments_are_unchanged_and_student_disjoint'],
    'scope':'portable software only; not main real-data Gate2 or CUDA readiness'},indent=2))
CPU_PORTABILITY_OK = True
print('CPU portability PASS',counts)


In [ ]:
# Chỉ chạy khi CPU gate đã đạt và source bundle vẫn giữ nguyên.
if not globals().get('CPU_PORTABILITY_OK',False):
    raise RuntimeError('CPU portability chưa PASS; không chạy GPU')
for path,expected in manifest['files'].items():
    if hashlib.sha256((ROOT / path).read_bytes()).hexdigest() != expected:
        raise ValueError('Source đổi sau CPU gate; không chạy GPU: '+path)
# Một synthetic epoch gồm một batch,400targets, timeout10phút.
# Probe từ chối device cuda nếu GPU không có, không tự fallback CPU.
RUN = ROOT / 'experiments/akt/results/colab-synthetic-cuda'
with (OUTPUT / 'cuda-probe.log').open('w') as log:
    result = subprocess.run([AKT_PY,'-B','scripts/akt/synthetic_device_probe.py',
                             '--device','cuda','--output-dir',str(RUN)],
                            cwd=ROOT,env=ENV,stdout=log,stderr=subprocess.STDOUT,timeout=600)
print((OUTPUT / 'cuda-probe.log').read_text()[-6000:])
result.check_returncode()
report = json.loads((RUN / 'report.json').read_text())
assert report['status'] == 'passed_synthetic' and report['cuda_lifecycle']
print({key:report[key] for key in ['status','gpu_name','peak_vram_bytes','training_seconds','gate4_pass','mid_epoch_resume_verified']})


In [ ]:
# Cell này vẫn có thể chạy thủ công khi test/probe fail để tải diagnostic logs.
import shutil
from google.colab import files
if 'RUN' in globals() and RUN.exists():
    shutil.copytree(RUN, OUTPUT / 'synthetic-run', dirs_exist_ok=True)
archive_path = shutil.make_archive(str(ROOT / 'akt-colab-results'), 'zip', OUTPUT)
files.download(archive_path)
